# Trace variants, payload profiling, and Pareto selection

This lab treats variant detection as a modeling decision. We compare multiple trace encodings and selection thresholds, then identify configurations that are non-dominated across four objectives:

- case coverage;
- compactness;
- decision relevance;
- temporal stability.

**Important:** a Pareto front produces a shortlist, not an automatic winner.

## 1. Setup

The lab uses a deterministic synthetic order-handling log. This lets us know which patterns and drift were introduced while still working with realistic lifecycle and payload fields.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

lab_dir = Path.cwd()
if not (lab_dir / "variant_pareto_lab.py").exists():
    lab_dir = Path("labs/01-variant-pareto")
sys.path.insert(0, str(lab_dir.resolve()))

from variant_pareto_lab import (
    build_case_table,
    encode_traces,
    evaluate_variant_candidates,
    generate_event_log,
    plot_pareto_candidates,
    profile_payload,
)

pd.set_option("display.max_colwidth", 100)

## 2. Inspect the event log

The minimum semantics are `case_id`, `activity`, and `timestamp`. Lifecycle transitions and payload enrich the analysis, but they also introduce representational choices.

In [ ]:
events = generate_event_log()
cases = build_case_table(events)

print(f"Events: {len(events):,}")
print(f"Cases: {events['case_id'].nunique():,}")
display(events.head(12))
display(cases.head())

### Checkpoint

1. Why do some activities have `suspend` and `resume` events?
2. Which columns are repeated event payload, and which are only known at completion?
3. Would filtering to completed cases change the question we can answer?

## 3. Profile the payload before encoding it

Payload is information carried by an event beyond its basic control-flow semantics. Missingness, cardinality, drift, privacy, and availability all matter before payload becomes part of a variant or machine-learning feature.

In [ ]:
payload_profile = profile_payload(events)
display(
    payload_profile.style.format(
        {"missing_rate": "{:.1%}", "drift_score": "{:.3f}"}
    )
)

### Leakage checkpoint

`outcome` is useful for evaluating whether variants distinguish operationally relevant populations, but it is only available at case completion. It must **not** be included in an online trace encoding.

Discuss what would happen if the outcome, final activity, or a completion-only document were included in a predictive prefix.

## 4. Compare trace encodings

Exact variant counts are produced by an encoding. They are not immutable properties of the source data.

In [ ]:
encoding_rows = []
traces_by_encoding = {}

for encoding in ["control_flow", "lifecycle", "time", "payload"]:
    traces = encode_traces(events, encoding)
    traces_by_encoding[encoding] = traces
    frequencies = traces.value_counts(normalize=True)
    encoding_rows.append(
        {
            "encoding": encoding,
            "exact_variants": traces.nunique(),
            "largest_variant_share": frequencies.iloc[0],
            "top_5_coverage": frequencies.head(5).sum(),
        }
    )

encoding_summary = pd.DataFrame(encoding_rows)
display(
    encoding_summary.style.format(
        {"largest_variant_share": "{:.1%}", "top_5_coverage": "{:.1%}"}
    )
)

In [ ]:
example_case = cases.loc[cases["path_name"] == "clarify", "case_id"].iloc[0]
print("Example case:", example_case)
for encoding, traces in traces_by_encoding.items():
    print(f"\n{encoding}:\n{traces.loc[example_case]}")

### Encoding checkpoint

- Which encoding fragments the population most aggressively?
- Which encoding preserves lifecycle interruptions?
- Does payload create meaningful cohorts, or merely many one-case variants?
- Which encoding would be defensible for the question you want to answer?

## 5. Evaluate variant-detection configurations

Each candidate combines:

- one trace encoding;
- an edit-similarity threshold (`1.00`, `0.80`, or `0.65`);
- a minimum variant frequency (`1`, `6`, or `20` cases).

All four objectives are maximized. Decision relevance uses the outcome only as an evaluation signal.

In [ ]:
candidates, labels_by_candidate = evaluate_variant_candidates(events)
metric_columns = [
    "candidate_id",
    "coverage",
    "compactness",
    "decision_relevance",
    "temporal_stability",
    "retained_variants",
    "pareto",
]
display(
    candidates[metric_columns].head(12).style.format(
        {
            "coverage": "{:.3f}",
            "compactness": "{:.3f}",
            "decision_relevance": "{:.3f}",
            "temporal_stability": "{:.3f}",
        }
    )
)

## 6. Inspect the Pareto front

The plot is a two-dimensional projection. Pareto membership is calculated using all four objectives, so a point can be non-dominated even when another point appears better in this projection.

In [ ]:
fig, axis = plot_pareto_candidates(candidates)
plt.show()

pareto_front = candidates[candidates["pareto"]]
display(pareto_front[metric_columns])

## 7. Apply constraints before preferences

Assume the stakeholders require at least 80% case coverage and 75% temporal stability. We first remove infeasible candidates, then inspect the remaining Pareto alternatives. Preferences come afterward.

In [ ]:
feasible_front = pareto_front.query(
    "coverage >= 0.80 and temporal_stability >= 0.75"
).copy()

feasible_front["stakeholder_view"] = (
    0.55 * feasible_front["decision_relevance"]
    + 0.45 * feasible_front["compactness"]
)

display(
    feasible_front.sort_values("stakeholder_view", ascending=False)[
        metric_columns + ["stakeholder_view"]
    ]
)

### Selection exercise

Choose one feasible configuration and write a short justification covering:

1. the analytical question;
2. why its encoding is meaningful;
3. what is gained and lost relative to an adjacent Pareto candidate;
4. representative traces you would inspect before approval;
5. payload, fairness, privacy, or leakage limitations;
6. how the configuration would be validated on a later time period.

## Optional PM4Py extension

If PM4Py is installed, convert the event table and compare its exact control-flow variants with the implementation used here:

```python
import pm4py

pm_log = pm4py.format_dataframe(
    events.query("lifecycle == 'complete'"),
    case_id='case_id',
    activity_key='activity',
    timestamp_key='timestamp',
)
pm4py.get_variants_as_tuples(pm_log)
```

Explain why lifecycle-, time-, and payload-aware variants require an explicit encoding decision before calling a variant-discovery function.